# AutoSense five-seed Bi-LSTM robustness (Kaggle)

This notebook executes the committed `multiseed_bilstm.py` without changing its seeds, architecture, chronological partitions, scaler policy, 20-epoch cap, batch size, or early-stopping rule. Attach the private Kaggle Dataset described in `research_v2/kaggle/README.md`.

In [ ]:
# Kaggle GPU/CPU environment dependencies. Restart the kernel if Kaggle requests it.
!pip -q install tensorflow==2.18.0 keras==3.15.1 numpy==2.0.2 pandas==2.2.2 scikit-learn==1.9.0
# Kaggle's preinstalled JAX requires an incompatible ml_dtypes version; this protocol does not use JAX.
!pip -q uninstall -y jax jaxlib
!nvidia-smi || true

In [ ]:
from pathlib import Path
import hashlib, shutil

INPUT = Path('/kaggle/input')
# A single uploaded kaggle_multiseed_input_bundle.zip is supported.
archives = list(INPUT.rglob('kaggle_multiseed_input_bundle.zip'))
if archives:
    assert len(archives) == 1, f'Expected one input bundle, found {archives}'
    import zipfile
    unpacked = Path('/kaggle/working/kaggle_input_bundle')
    if unpacked.exists(): shutil.rmtree(unpacked)
    with zipfile.ZipFile(archives[0]) as z: z.extractall(unpacked)
    INPUT = unpacked
candidates = [p for p in INPUT.rglob('multiseed_bilstm.py') if p.parent.name == 'ieee_experiments' and p.parent.parent.name == 'research_v2']
assert len(candidates) == 1, f'Attach exactly one prepared input Dataset; found {candidates}'
SOURCE_ROOT = candidates[0].parents[2]
WORK = Path('/kaggle/working/autosense_ieee')
if WORK.exists(): shutil.rmtree(WORK)
shutil.copytree(SOURCE_ROOT, WORK)
expected = {
 'FYP ML Training and Pipeline Code/Engine Temperature/karachi_2024_vehicle_temperature_data.csv': 'd6528d327f13978592bff712684343087a99942694d99f81ebda7ff28978581e',
 'FYP ML Training and Pipeline Code/TP_Generated/tyre_pressure_2024.csv': '63270ee9d3211afc3b6c6fcca798b89c4303721caa6924c34e44d112d45688f3',
 'FYP ML Training and Pipeline Code/Tyre Temperature/tyre_temp_2024.csv': '451301b1fef1c683e844bffdec67cc93c607836b42ce1a344219b7b080379c3a'}
for relative, wanted in expected.items():
    actual = hashlib.sha256((WORK/relative).read_bytes()).hexdigest()
    assert actual == wanted, f'hash mismatch: {relative}: {actual}'
print('Validated exact input data hashes; writable project root:', WORK)

In [ ]:
%cd /kaggle/working/autosense_ieee
!python -u research_v2/ieee_experiments/multiseed_bilstm.py --verbose 2
# A fresh Kaggle run deliberately does not use --resume: it must contain all five seeds.

In [ ]:
import pandas as pd, zipfile
results = WORK/'research_v2'/'ieee_results'
metrics = pd.read_csv(results/'multiseed_bilstm_metrics.csv')
summary = pd.read_csv(results/'multiseed_summary.csv')
assert len(metrics) == 15 and set(metrics.Seed) == {11, 21, 42, 84, 168}
assert len(summary) == 3 and summary['Completed planned seeds'].all()
display(summary[['Dataset', 'MAE mean', 'MAE std', 'RMSE mean', 'RMSE std', 'R2 mean', 'R2 std']])
archive = Path('/kaggle/working/autosense_multiseed_results.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(results/'multiseed_bilstm_metrics.csv', 'multiseed_bilstm_metrics.csv')
    z.write(results/'multiseed_summary.csv', 'multiseed_summary.csv')
print('Download:', archive)